# 10 - Dashboard Preparation

## Electricity Consumption Analysis and Forecasting

### Objective

Prepare the final datasets and summary values needed to build the project's interactive dashboard.

The dashboard will combine:

- Historical electricity consumption
- Hourly consumption patterns
- Weekday vs weekend analysis
- Monthly patterns
- Peak consumption periods
- Actual vs predicted consumption
- Model performance
- Next-hour forecasting

This notebook focuses on preparing clean dashboard-ready files. The actual Streamlit application will be created after this notebook.


## 1. Import Libraries


In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

pd.set_option("display.max_columns", None)


## 2. Load Main Datasets


In [2]:
hourly_path = "../data/processed/hourly_electricity_consumption.csv"
prediction_path = "../data/processed/test_predictions.csv"
model_results_path = "../data/processed/model_comparison_results.csv"

hourly_df = pd.read_csv(
    hourly_path,
    parse_dates=["DateTime"]
).set_index("DateTime").sort_index()

prediction_df = pd.read_csv(
    prediction_path,
    parse_dates=["DateTime"]
).set_index("DateTime").sort_index()

model_results = pd.read_csv(model_results_path)

print("Hourly data:", hourly_df.shape)
print("Prediction data:", prediction_df.shape)
print("Model results:", model_results.shape)


Hourly data: (24584, 15)
Prediction data: (4883, 9)
Model results: (4, 4)


## 3. Prepare Hourly Pattern Data


In [3]:
hourly_pattern = (
    hourly_df
    .groupby("Hour")["Electricity_Consumption"]
    .mean()
    .reset_index()
)

hourly_pattern.columns = [
    "Hour",
    "Average_Consumption"
]

display(hourly_pattern)


,Hour,Average_Consumption
0,0,0.662266
1,1,0.534260
2,2,0.469880
3,3,0.443495
4,4,0.446498
5,5,0.458132
6,6,0.810217
7,7,1.481431
8,8,1.457027
9,9,1.308342


## 4. Prepare Day-of-Week Data


In [4]:
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

day_pattern = (
    hourly_df
    .groupby("DayName")["Electricity_Consumption"]
    .mean()
    .reindex(day_order)
    .reset_index()
)

day_pattern.columns = [
    "DayName",
    "Average_Consumption"
]

display(day_pattern)


,DayName,Average_Consumption
0,Monday,0.979632
1,Tuesday,1.066451
2,Wednesday,1.061840
3,Thursday,0.945995
4,Friday,1.042660
5,Saturday,1.248035
6,Sunday,1.238904


## 5. Prepare Weekday vs Weekend Data


In [5]:
weekend_pattern = (
    hourly_df
    .groupby("IsWeekend")["Electricity_Consumption"]
    .mean()
)

weekend_pattern.index = [
    "Weekday" if value == 0 else "Weekend"
    for value in weekend_pattern.index
]

weekend_pattern = (
    weekend_pattern
    .rename("Average_Consumption")
    .reset_index()
)

display(weekend_pattern)


,index,Average_Consumption
0,Weekday,1.019264
1,Weekend,1.243459


## 6. Prepare Monthly Data


In [6]:
month_pattern = (
    hourly_df
    .groupby("Month")["Electricity_Consumption"]
    .mean()
    .reset_index()
)

month_pattern.columns = [
    "Month",
    "Average_Consumption"
]

display(month_pattern)


,Month,Average_Consumption
0,1,1.472121
1,2,1.275429
2,3,1.263607
3,4,1.034074
4,5,1.007663
5,6,0.897804
6,7,0.693767
7,8,0.565694
8,9,0.981352
9,10,1.109701


## 7. Prepare Top Consumption Periods


In [7]:
top_consumption = (
    hourly_df[
        ["Electricity_Consumption"]
    ]
    .sort_values(
        "Electricity_Consumption",
        ascending=False
    )
    .head(20)
    .reset_index()
)

display(top_consumption.head(10))


,DateTime,Electricity_Consumption
0,2008-11-23 18:00:00,6.560533
1,2009-01-16 20:00:00,6.519633
2,2008-02-02 19:00:00,6.496033
3,2007-12-23 19:00:00,6.488000
4,2007-02-22 21:00:00,6.363867
5,2007-12-28 17:00:00,6.333667
6,2008-11-23 20:00:00,6.310567
7,2007-01-21 20:00:00,6.076567
8,2008-01-26 19:00:00,6.013800
9,2008-11-30 20:00:00,5.930500


## 8. Prepare Prediction Data


In [8]:
prediction_dashboard = prediction_df[
    [
        "Actual",
        "Predicted",
        "Baseline",
        "Error",
        "Absolute_Error"
    ]
].reset_index()

prediction_dashboard["Hour"] = (
    prediction_dashboard["DateTime"].dt.hour
)

prediction_dashboard["DayName"] = (
    prediction_dashboard["DateTime"].dt.day_name()
)

prediction_dashboard["Month"] = (
    prediction_dashboard["DateTime"].dt.month
)

display(prediction_dashboard.head())


,DateTime,Actual,Predicted,Baseline,Error,Absolute_Error,Hour,DayName,Month
0,2009-03-16 13:00:00,0.266300,1.015103,1.351733,-0.748803,0.748803,13,Monday,3
1,2009-03-16 14:00:00,0.277667,1.161157,1.019467,-0.883490,0.883490,14,Monday,3
2,2009-03-16 15:00:00,0.315533,0.495002,0.266300,-0.179468,0.179468,15,Monday,3
3,2009-03-16 16:00:00,0.297867,0.555103,0.277667,-0.257237,0.257237,16,Monday,3
4,2009-03-16 17:00:00,0.318733,1.091982,0.315533,-0.773248,0.773248,17,Monday,3


## 9. Prepare Model Performance Data


In [9]:
model_performance = model_results.copy()

model_performance = model_performance.sort_values(
    "MAE"
).reset_index(drop=True)

display(model_performance)


,Model,MAE,RMSE,R2
0,HistGradientBoosting,0.374329,0.518418,0.455438
1,Random Forest,0.387896,0.533059,0.424245
2,Linear Regression,0.466991,0.618019,0.226089
3,Previous-Hour Baseline,0.534101,0.795210,-0.281300


## 10. Calculate Dashboard KPIs


In [10]:
best_ml_results = model_results[
    model_results["Model"] != "Previous-Hour Baseline"
].sort_values("MAE")

best_model_row = best_ml_results.iloc[0]

baseline_row = model_results[
    model_results["Model"] == "Previous-Hour Baseline"
].iloc[0]

best_model_name = best_model_row["Model"]

mae_improvement = (
    (baseline_row["MAE"] - best_model_row["MAE"])
    / baseline_row["MAE"]
    * 100
)

rmse_improvement = (
    (baseline_row["RMSE"] - best_model_row["RMSE"])
    / baseline_row["RMSE"]
    * 100
)

dashboard_kpis = {
    "Best Model": best_model_name,
    "Best Model MAE": float(best_model_row["MAE"]),
    "Best Model RMSE": float(best_model_row["RMSE"]),
    "Best Model R2": float(best_model_row["R2"]),
    "Baseline MAE": float(baseline_row["MAE"]),
    "Baseline RMSE": float(baseline_row["RMSE"]),
    "Baseline R2": float(baseline_row["R2"]),
    "MAE Improvement Percent": float(mae_improvement),
    "RMSE Improvement Percent": float(rmse_improvement),
    "Average Consumption": float(
        hourly_df["Electricity_Consumption"].mean()
    ),
    "Maximum Hourly Consumption": float(
        hourly_df["Electricity_Consumption"].max()
    ),
    "Minimum Hourly Consumption": float(
        hourly_df["Electricity_Consumption"].min()
    )
}

dashboard_kpis


{'Best Model': 'HistGradientBoosting',
 'Best Model MAE': 0.374329406315359,
 'Best Model RMSE': 0.5184178276691596,
 'Best Model R2': 0.4554382990233575,
 'Baseline MAE': 0.534100681393319,
 'Baseline RMSE': 0.7952098232328327,
 'Baseline R2': -0.2812996599045363,
 'MAE Improvement Percent': 29.914074376606585,
 'RMSE Improvement Percent': 34.80741654302101,
 'Average Consumption': 1.0834565776513523,
 'Maximum Hourly Consumption': 6.560533333333334,
 'Minimum Hourly Consumption': 0.124}

## 11. Find Peak Consumption Information


In [11]:
peak_row = (
    hourly_df[
        ["Electricity_Consumption"]
    ]
    .sort_values(
        "Electricity_Consumption",
        ascending=False
    )
    .iloc[0]
)

peak_timestamp = (
    hourly_df["Electricity_Consumption"]
    .idxmax()
)

dashboard_kpis["Peak Consumption"] = float(
    peak_row["Electricity_Consumption"]
)

dashboard_kpis["Peak Consumption Time"] = (
    str(peak_timestamp)
)

print("Peak consumption:", dashboard_kpis["Peak Consumption"])
print("Peak timestamp:", dashboard_kpis["Peak Consumption Time"])


Peak consumption: 6.560533333333334
Peak timestamp: 2008-11-23 18:00:00


## 12. Save Dashboard Datasets


In [12]:
dashboard_dir = Path("../data/processed/dashboard")
dashboard_dir.mkdir(parents=True, exist_ok=True)

hourly_pattern.to_csv(
    dashboard_dir / "hourly_pattern.csv",
    index=False
)

day_pattern.to_csv(
    dashboard_dir / "day_pattern.csv",
    index=False
)

weekend_pattern.to_csv(
    dashboard_dir / "weekend_pattern.csv",
    index=False
)

month_pattern.to_csv(
    dashboard_dir / "month_pattern.csv",
    index=False
)

top_consumption.to_csv(
    dashboard_dir / "top_consumption.csv",
    index=False
)

prediction_dashboard.to_csv(
    dashboard_dir / "prediction_dashboard.csv",
    index=False
)

model_performance.to_csv(
    dashboard_dir / "model_performance.csv",
    index=False
)

with open(
    dashboard_dir / "dashboard_kpis.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        dashboard_kpis,
        file,
        indent=4
    )

print("Dashboard files saved to:", dashboard_dir)


Dashboard files saved to: ..\data\processed\dashboard


## 13. Verify Dashboard Files


In [13]:
for file in sorted(dashboard_dir.iterdir()):
    print("-", file.name)


- dashboard_kpis.json
- day_pattern.csv
- hourly_pattern.csv
- model_performance.csv
- month_pattern.csv
- prediction_dashboard.csv
- top_consumption.csv
- weekend_pattern.csv


## 14. Create a Compact Dashboard Summary


In [14]:
dashboard_summary = pd.DataFrame({
    "Metric": [
        "Best ML Model",
        "Best Model MAE",
        "Best Model RMSE",
        "Best Model R²",
        "MAE Improvement vs Baseline (%)",
        "RMSE Improvement vs Baseline (%)",
        "Average Consumption",
        "Maximum Hourly Consumption",
        "Minimum Hourly Consumption"
    ],
    "Value": [
        dashboard_kpis["Best Model"],
        dashboard_kpis["Best Model MAE"],
        dashboard_kpis["Best Model RMSE"],
        dashboard_kpis["Best Model R2"],
        dashboard_kpis["MAE Improvement Percent"],
        dashboard_kpis["RMSE Improvement Percent"],
        dashboard_kpis["Average Consumption"],
        dashboard_kpis["Maximum Hourly Consumption"],
        dashboard_kpis["Minimum Hourly Consumption"]
    ]
})

display(dashboard_summary)


,Metric,Value
0,Best ML Model,HistGradientBoosting
1,Best Model MAE,0.374329
2,Best Model RMSE,0.518418
3,Best Model R²,0.455438
4,MAE Improvement vs Baseline (%),29.914074
5,RMSE Improvement vs Baseline (%),34.807417
6,Average Consumption,1.083457
7,Maximum Hourly Consumption,6.560533
8,Minimum Hourly Consumption,0.124


## 15. Final Validation


In [15]:
print("===== DASHBOARD PREPARATION SUMMARY =====")
print("Best model:", dashboard_kpis["Best Model"])
print("Best model MAE:", dashboard_kpis["Best Model MAE"])
print("Best model RMSE:", dashboard_kpis["Best Model RMSE"])
print("Best model R²:", dashboard_kpis["Best Model R2"])
print("MAE improvement:", dashboard_kpis["MAE Improvement Percent"], "%")
print("RMSE improvement:", dashboard_kpis["RMSE Improvement Percent"], "%")
print("Peak consumption:", dashboard_kpis["Peak Consumption"])
print("Peak timestamp:", dashboard_kpis["Peak Consumption Time"])

print("\nDashboard files:")
for file in sorted(dashboard_dir.iterdir()):
    print("-", file.name)


===== DASHBOARD PREPARATION SUMMARY =====
Best model: HistGradientBoosting
Best model MAE: 0.374329406315359
Best model RMSE: 0.5184178276691596
Best model R²: 0.4554382990233575
MAE improvement: 29.914074376606585 %
RMSE improvement: 34.80741654302101 %
Peak consumption: 6.560533333333334
Peak timestamp: 2008-11-23 18:00:00

Dashboard files:
- dashboard_kpis.json
- day_pattern.csv
- hourly_pattern.csv
- model_performance.csv
- month_pattern.csv
- prediction_dashboard.csv
- top_consumption.csv
- weekend_pattern.csv


## Conclusion

The project data is now prepared for an interactive dashboard.

### Dashboard sections

The final dashboard should contain:

1. **Overview**
   - Average consumption
   - Peak consumption
   - Minimum consumption
   - Best ML model

2. **Consumption Analysis**
   - Hourly pattern
   - Day-of-week pattern
   - Weekday vs weekend
   - Monthly pattern
   - Top consumption periods

3. **Forecasting**
   - Actual vs predicted
   - Model performance
   - Next-hour forecast

4. **Model Evaluation**
   - MAE
   - RMSE
   - R²
   - Baseline comparison

### Next file

**`11_streamlit_dashboard.py`**

This will be the first non-notebook application file. It will create the interactive web dashboard for the college project.
